In [17]:
import parser
import solution
import mapper

# tường '#'
# đích '.'
# thùng '$'
# thùng trên đích '*'

level = parser.parse_level("levels/competition_levels/MANotHard.lvl")
level.print_state()
level.obstacles |= mapper.compute_dead_squares(level)
level.print_state()

ACTION was START
##########################
##########################
## . $ b          h $ . ##
## . $ c ###   ## i $ . ##
## . $ d ######## j $ . ##
##                      ##
##    .              .  ##
##         $$$$         ##
##        $             ##
##    .             .   ##
##          .           ##
## . $ e          a $ . ##
## . $ f ########   $ . ##
## . $ g            $ . ##
##########################
##########################

ACTION was START
##########################
##########################
###. $ b          h $ .###
###. $ c ######## i $ .###
###. $ d ######## j $ .###
###                    ###
###   .              . ###
###        $$$$        ###
###       $            ###
###   .             .  ###
###         .          ###
###. $ e          a $ .###
###. $ f ########   $ .###
###. $ g            $ .###
##########################
##########################



In [2]:
from sokoban import PROBLEMS
import AStarW
import RobotHeuristic as RH
import mapper
import heapq
from sokoban import sokoban_goal_state
import time

p = PROBLEMS[9]
print('Initial h:', RH.combined_heuristic(p, mode='guided'))
print('Initial box_to_storage:', RH.box_to_storage_heuristic(p))
print('Initial robot_to_box:', RH.robot_to_box_heuristic(p, mode='guided'))

# Let us run AStarW with step-by-step progress logging
t0 = time.time()
pq = []
initial_state = p
w = 2
h_init = RH.combined_heuristic(initial_state, mode='guided')
heapq.heappush(pq, AStarW.Node(initial_state, initial_state.gval + w * h_init, h_init))
explored = {initial_state.hashable_state(): initial_state.gval}
map_info = mapper.get_map_info(initial_state)

pop_count = 0
last_t = time.time()
while pq and time.time() - t0 < 5.0:
    curr_node = heapq.heappop(pq)
    curr = curr_node.state
    pop_count += 1

    if time.time() - last_t >= 1.0:
        pop_count = 0
        last_t = time.time()
        
while pq and time.time() - t0 < 5.0:
    curr_node = heapq.heappop(pq)
    curr = curr_node.state
    pop_count += 1

    if time.time() - last_t >= 1.0:
        print(f'T={time.time()-t0:.1f}s, PopCount={pop_count}, Frontier={len(pq)}, Explored={len(explored)}, Curr gval={curr.gval},f={curr_node.f_value:.1f}, h={curr_node.h_value:.1f}')

    if sokoban_goal_state(curr):
        print('SOLVED!')
        break

    for next_state in curr.successors():
        next_key = next_state.hashable_state()
        new_g = next_state.gval
        if next_key not in explored or new_g < explored[next_key]:
            if mapper.is_deadlock(next_state, map_info):
                continue
            h_val = RH.combined_heuristic(next_state, mode='guided')
            if h_val == float('inf'):
                continue
            explored[next_key] = new_g
            f_val = new_g + w * h_val
            f_val = new_g + w * h_val
            heapq.heappush(pq, AStarW.Node(next_state, f_val, h_val))

print(f'Done 5s test: PopCount={pop_count}, Explored={len(explored)}, Remaining Frontier={len(pq)}')

Initial h: 12
Initial box_to_storage: 12
Initial robot_to_box: 0
Done 5s test: PopCount=1, Explored=1, Remaining Frontier=0
